In [ ]:
# %% [0] Instellingen en functies (eerst uitvoeren)
import csv
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

In [ ]:
# cpi data
cpi_file = os.path.join("..", "data", "processed_data", 'cpi_data.csv')
cpi_df = pd.read_csv(cpi_file, delimiter=';', encoding='utf-8', index_col=[0], parse_dates=True)

# mode data
mode_file = os.path.join("..", "data", "processed_data", 'mode_data.csv')
mode_df = pd.read_csv(mode_file, delimiter=';', encoding='utf-8', header=[0, 1], index_col=[0, 1], parse_dates=True)

# display dataframes
display(cpi_df)
display(mode_df)

In [ ]:
OUT = Path("figures")

# !! TIJDELIJKE gewichten: vervang door de echte aandelen uit de CLO-data (tellen op tot 1)
WEIGHTS = {"CPI_Diesel": 1 / 3, "CPI_Petrol": 1 / 3, "CPI_LPG": 1 / 3}

LAG = 0                 # reactietijd in jaren: 0 = zelfde jaar, 1 = prijsratio van vorig jaar
EXCLUDE_COVID = True    # COVID-jaren weglaten bij de elasticiteitsschatting
COVID = (2020, 2021)

COL = {"train": "#E07A2F", "car": "#1F3A5F", "diesel": "#6B7280", "petrol": "#6C8EBF",
       "lpg": "#9CA3AF", "ratio": "#B4432F"}

plt.rcParams.update({
    "figure.figsize": (9, 5.2), "figure.dpi": 110, "savefig.dpi": 200, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "axes.titleweight": "bold", "axes.titlesize": 14,
    "axes.titlelocation": "left",
})


def save(fig, name):
    OUT.mkdir(exist_ok=True)
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight")


def covid_band(ax, label=True):
    ax.axvspan(COVID[0] - 0.5, COVID[1] + 0.5, color="#9CA3AF", alpha=0.15, lw=0)
    if label:
        ax.text(sum(COVID) / 2, ax.get_ylim()[1], "COVID-19", ha="center", va="top",
                fontsize=9, color="#6B7280")


In [ ]:


# %% [1] Data inlezen en samenvoegen
# cpi data
cpi_file = os.path.join("..", "data", "processed_data", 'cpi_data.csv')
cpi_df = pd.read_csv(cpi_file, delimiter=';', encoding='utf-8', index_col=[0], parse_dates=True)

# mode data
mode_file = os.path.join("..", "data", "processed_data", 'mode_data.csv')
mode_df = pd.read_csv(mode_file, delimiter=';', encoding='utf-8', header=[0, 1], index_col=[0, 1], parse_dates=True)


In [ ]:

# display dataframes
try:
    display(cpi_df)
    display(mode_df)
except NameError:      # buiten Jupyter/Interactive Window bestaat display() niet
    print(cpi_df.head())
    print(mode_df.head())


In [ ]:

# --- CPI (maandelijks): brandstofindex en prijsratio ---
cpi = cpi_df.copy()
cpi["CPI_Fuel"] = sum(cpi[k] * w for k, w in WEIGHTS.items())
cpi["Ratio"] = cpi["CPI_Train"] / cpi["CPI_Fuel"]          # fare-to-fuel
cpi_year = cpi.groupby(cpi.index.year).mean()
cpi_year.index.name = "year"


In [ ]:

# --- Mode data (jaarlijks, per reismotief): jaar als index + korte indicatornamen ---
short = {
    "Distance per person per day (km)": "km_day",
    "Distance per trip (km)": "km_trip",
    "Travel time per person per day (min)": "min_day",
    "Travel time per trip (min)": "min_trip",
    "Trips per person per day (trips)": "trips_day",
}
mode = mode_df.copy()
mode.columns = pd.MultiIndex.from_tuples([(m, short[i]) for m, i in mode_df.columns],
                                         names=["mode", "indicator"])
mode.index = pd.MultiIndex.from_arrays(
    [mode_df.index.get_level_values(0).year, mode_df.index.get_level_values(1)],
    names=["year", "purpose"])


def get(m, indicator, purpose="Total"):
    return mode.xs(purpose, level="purpose")[(m, indicator)]



In [ ]:

# --- Hoofddataset (alle reismotieven samen) ---
df = pd.DataFrame({
    "train_km": get("Train", "km_day"), "car_km": get("Car (total)", "km_day"),
    "train_trips": get("Train", "trips_day"), "car_trips": get("Car (total)", "trips_day"),
})
df["train_share"] = df["train_km"] / (df["train_km"] + df["car_km"]) * 100
df = df.join(cpi_year[["CPI_Train", "CPI_Fuel", "Ratio"]], how="inner")
print("Modes:", sorted(set(mode.columns.get_level_values(0))))
print("Reismotieven:", list(mode.index.get_level_values("purpose").unique()))
df.round(3).head()



In [ ]:

# %% [2] Grafiek 1: prijsindexen (2015 = 100)
fig, ax = plt.subplots()
ax.plot(cpi.index, cpi["CPI_Train"], color=COL["train"], lw=2.8, label="Train fare")
ax.plot(cpi.index, cpi["CPI_Petrol"], color=COL["petrol"], lw=1.6, label="Petrol")
ax.plot(cpi.index, cpi["CPI_Diesel"], color=COL["diesel"], lw=1.6, label="Diesel")
ax.plot(cpi.index, cpi["CPI_LPG"], color=COL["lpg"], lw=1.4, label="LPG")
ax.axhline(100, color="#9CA3AF", lw=1, ls="--")
ax.set_title("Consumer price index: train fare vs fuel (2015 = 100)")
ax.set_ylabel("CPI")
ax.legend(frameon=False, ncol=4, loc="upper left")
save(fig, "1_cpi_train_vs_fuel")
plt.show()


In [ ]:


# %% [3] Grafiek 2: fare-to-fuel ratio
fig, ax = plt.subplots()
ax.plot(cpi.index, cpi["Ratio"], color=COL["ratio"], lw=1, alpha=0.35, label="Monthly")
ax.plot(pd.to_datetime(cpi_year.index.astype(str) + "-07-01"), cpi_year["Ratio"],
        color=COL["ratio"], lw=2.8, label="Annual mean")
top, bottom = cpi_year["Ratio"].idxmax(), cpi_year["Ratio"].idxmin()
for y in (top, bottom):
    ax.annotate(f"{y}: {cpi_year.loc[y, 'Ratio']:.2f}", (pd.Timestamp(f"{y}-07-01"), cpi_year.loc[y, "Ratio"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
ax.axhline(1, color="#9CA3AF", lw=1, ls="--")
ax.set_title("Fare-to-fuel price ratio (higher = train relatively more expensive)")
ax.set_ylabel("Train CPI / fuel CPI")
ax.legend(frameon=False)
save(fig, "2_fare_fuel_ratio")
plt.show()



In [ ]:

# %% [4] Grafiek 3: modal split (aandeel van auto + trein km)
fig, ax = plt.subplots()
share_car = 100 - df["train_share"]
ax.stackplot(df.index, share_car, df["train_share"], labels=["Car", "Train"],
             colors=[COL["car"], COL["train"]], alpha=0.9)
ax.set_ylim(0, 100)
ax.set_xlim(df.index.min(), df.index.max())
ax.set_ylabel("% of car + train kilometres")
ax.set_title("Modal split car vs train (share of distance travelled)")
ax.legend(loc="lower left", frameon=False)
for y in (df.index.min(), df.index.max()):
    ax.text(y, share_car[y] + 1.2, f"{df.loc[y, 'train_share']:.1f}% train", color="white", fontsize=9,
            ha="left" if y == df.index.min() else "right")
save(fig, "3_modal_split")
plt.show()



In [ ]:

# %% [5] Grafiek 4: prijsratio en treinaandeel in de tijd (twee assen)
fig, ax1 = plt.subplots()
ax1.plot(df.index, df["Ratio"], color=COL["ratio"], lw=2.6, label="Fare-to-fuel ratio")
ax1.set_ylabel("Fare-to-fuel ratio", color=COL["ratio"])
ax2 = ax1.twinx()
ax2.plot(df.index, df["train_share"], color=COL["train"], lw=2.6, label="Train share of km")
ax2.set_ylabel("Train share of car + train km (%)", color=COL["train"])
ax2.grid(False)
ax2.spines["right"].set_visible(True)
covid_band(ax1)
ax1.set_title("Price ratio and train share over time")
save(fig, "4_ratio_en_treinaandeel")
plt.show()



In [ ]:

# %% [6] Grafiek 5: scatter prijsratio vs treinaandeel (kernplaatje voor je onderzoeksvraag)
d = df.dropna(subset=["Ratio", "train_share"])
fig, ax = plt.subplots(figsize=(8.5, 5.6))
sc = ax.scatter(d["Ratio"], d["train_share"], c=d.index, cmap="viridis", s=70,
                edgecolor="white", linewidth=1, zorder=3)
cb = fig.colorbar(sc, ax=ax, pad=0.02)
cb.set_label("Year")
fit = d.drop(index=[y for y in range(COVID[0], COVID[1] + 1) if y in d.index]) if EXCLUDE_COVID else d
k, b = np.polyfit(fit["Ratio"], fit["train_share"], 1)
xs = np.linspace(d["Ratio"].min(), d["Ratio"].max(), 50)
ax.plot(xs, k * xs + b, color=COL["ratio"], lw=2, ls="--",
        label=f"Linear fit{' (excl. COVID)' if EXCLUDE_COVID else ''}: slope {k:.1f} pp per ratio point")
for y in (d.index.min(), d.index.max(), *COVID):
    if y in d.index:
        ax.annotate(str(y), (d.loc[y, "Ratio"], d.loc[y, "train_share"]), textcoords="offset points",
                    xytext=(6, 6), fontsize=9)
ax.set_xlabel("Fare-to-fuel ratio")
ax.set_ylabel("Train share of car + train km (%)")
ax.set_title("Does a higher relative train price lower the train share?")
ax.legend(frameon=False, loc="best")
save(fig, "5_scatter_ratio_vs_share")
plt.show()



In [ ]:

# %% [7] Elasticiteit (log-log): hoeveel % verandert het gebruik bij 1% hogere ratio?
def elasticity(y, x, lag=LAG, exclude_covid=EXCLUDE_COVID):
    """Helling van ln(y) op ln(x). x wordt 'lag' jaar vertraagd. Geeft (e, r2, n, p_approx)."""
    t = pd.concat([y, x.shift(lag)], axis=1, keys=["y", "x"]).dropna()
    if exclude_covid:
        t = t.drop(index=[j for j in range(COVID[0], COVID[1] + 1) if j in t.index])
    t = t[(t > 0).all(axis=1)]
    lx, ly = np.log(t["x"]), np.log(t["y"])
    e, c = np.polyfit(lx, ly, 1)
    resid = ly - (e * lx + c)
    r2 = 1 - resid.var() / ly.var()
    n = len(t)
    se = np.sqrt(resid.var(ddof=2) / ((lx - lx.mean()) ** 2).sum())
    return {"elasticity": e, "se": se, "r2": r2, "n": n, "intercept": c, "x": lx, "y": ly}


res = {name: elasticity(df[col], df["Ratio"]) for name, col in
       [("Train km", "train_km"), ("Car km", "car_km"), ("Train share", "train_share")]}
tabel = pd.DataFrame({k: {"elasticity": v["elasticity"], "std. error": v["se"],
                          "R2": v["r2"], "n": v["n"]} for k, v in res.items()}).T
print(f"LAG={LAG}, EXCLUDE_COVID={EXCLUDE_COVID}")
print(tabel.round(3))
print("\nLet op: tijdreeksen met trends geven vaak schijnverbanden; een lage R2 of hoge std. error"
      " betekent dat je de elasticiteit voorzichtig moet interpreteren.")



In [ ]:

# %% [8] Grafiek 6: log-log regressie voor trein en auto
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, name, color in zip(axes, ["Train km", "Car km"], [COL["train"], COL["car"]]):
    r = res[name]
    ax.scatter(r["x"], r["y"], color=color, s=55, edgecolor="white", zorder=3)
    xs = np.linspace(r["x"].min(), r["x"].max(), 50)
    ax.plot(xs, r["elasticity"] * xs + r["intercept"], color="black", lw=1.8, ls="--")
    ax.set_title(f"{name}: elasticity {r['elasticity']:.2f} (R² {r['r2']:.2f})", fontsize=12)
    ax.set_xlabel("ln(fare-to-fuel ratio)")
    ax.set_ylabel(f"ln({name.lower()} per person per day)")
save(fig, "6_loglog_elasticiteit")
plt.show()



In [ ]:

# %% [9] Grafiek 7: elasticiteit per reismotief (treinkilometers)
purposes = [p for p in mode.index.get_level_values("purpose").unique()]
rows = []
for p in purposes:
    r = elasticity(get("Train", "km_day", p), df["Ratio"])
    rows.append((p, r["elasticity"], 1.96 * r["se"]))
pur = pd.DataFrame(rows, columns=["purpose", "e", "ci"]).sort_values("e")

fig, ax = plt.subplots()
ax.barh(pur["purpose"], pur["e"], xerr=pur["ci"], color=COL["train"], alpha=0.9,
        error_kw={"ecolor": "#4B5563", "capsize": 4})
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Elasticity of train km w.r.t. fare-to-fuel ratio (± 95% CI, rough)")
ax.set_title("Price sensitivity of train use by travel purpose")
ax.grid(axis="y", visible=False)
save(fig, "7_elasticiteit_per_reismotief")
plt.show()



In [ ]:

# %% [10] Export voor je project
export = df.copy()
export.to_csv(OUT / "project_dataset.csv", sep=";", decimal=",")
tabel.to_csv(OUT / "elasticiteit_resultaten.csv", sep=";", decimal=",")
print("Opgeslagen in", OUT.resolve())